In [2]:
import json
import re
from pathlib import Path


# ============================================================
# DATEIPFADE
# ============================================================

ORDNER = Path(
    r"P:\pcloud\Projekte_pcloud\isochrone_map"
)

EINGABEDATEI = ORDNER / "verkehrszeichen.json"

AUSGABEDATEI = (
    ORDNER / "geschwindigkeitsrelevante_verkehrszeichen.json"
)


# ============================================================
# RELEVANTE VERKEHRSZEICHEN
# ============================================================

# Direkte Geschwindigkeitsbegrenzungen
GESCHWINDIGKEITSZEICHEN = {
    "274",      # Zulässige Höchstgeschwindigkeit
    "274.1",    # Beginn einer Tempo-Zone
    "274.2",    # Ende einer Tempo-Zone
    "278",      # Ende der zulässigen Höchstgeschwindigkeit
}

# Fahrradstraße
FAHRRADSTRASSEN = {
    "244.1",    # Beginn einer Fahrradstraße
    "244.2",    # Ende einer Fahrradstraße
}


# ============================================================
# HILFSFUNKTIONEN
# ============================================================

def nummer_normalisieren(vz_nr):
    """
    Vereinheitlicht die Verkehrszeichennummer.

    Beispiele:
        274     -> "274"
        "274"   -> "274"
        "274.1" -> "274.1"
    """

    if vz_nr is None:
        return ""

    return str(vz_nr).strip()


def geschwindigkeit_finden(properties):
    """
    Versucht, eine Geschwindigkeitsangabe aus den vorhandenen
    Attributen zu extrahieren.

    Durchsucht verschiedene Textfelder.
    """

    # Besonders relevante Felder
    relevante_felder = [
        "vz_bez",
        "vz_zus_tx",
        "vz_var",
        "vz_gr",
    ]

    for feld in relevante_felder:

        wert = properties.get(feld)

        if wert is None:
            continue

        text = str(wert)

        # Suche nach typischen Geschwindigkeiten
        #
        # Beispiele:
        # "30"
        # "Tempo 30"
        # "30 km/h"
        # "Höchstgeschwindigkeit 50"

        treffer = re.search(
            r"\b(5|10|20|30|40|50|60|70|80|90|100|120|130)\b",
            text
        )

        if treffer:

            return int(
                treffer.group(1)
            )

    return None


def ist_geschwindigkeitsrelevant(properties):
    """
    Prüft, ob ein Verkehrszeichen für die Geschwindigkeit
    oder die Art der Fortbewegung relevant ist.
    """

    vz_nr = nummer_normalisieren(
        properties.get("vz_nr")
    )

    vz_bez = str(
        properties.get("vz_bez") or ""
    ).lower()

    vz_zusatz = str(
        properties.get("vz_zus_tx") or ""
    ).lower()

    gesamter_text = (
        vz_bez
        + " "
        + vz_zusatz
    )


    # --------------------------------------------------------
    # Direkte Verkehrszeichennummern
    # --------------------------------------------------------

    if vz_nr in GESCHWINDIGKEITSZEICHEN:
        return True

    if vz_nr in FAHRRADSTRASSEN:
        return True


    # --------------------------------------------------------
    # Suche über die Bezeichnung
    # --------------------------------------------------------

    relevante_begriffe = [

        # Geschwindigkeit
        "höchstgeschwindigkeit",
        "geschwindigkeit",
        "tempo",

        # Fahrradstraße
        "fahrradstraße",
        "fahrradstrasse",

        # Verkehrsberuhigung
        "verkehrsberuhigter bereich",

        # Schrittgeschwindigkeit
        "schrittgeschwindigkeit",

        # Fußgängerbereiche
        "fußgängerbereich",
        "fussgängerbereich",

        # Spielstraße
        "spielstraße",
        "spielstrasse",
    ]


    for begriff in relevante_begriffe:

        if begriff in gesamter_text:
            return True


    return False


def regelung_bestimmen(properties):
    """
    Bestimmt die Art der Geschwindigkeitsregelung.
    """

    vz_nr = nummer_normalisieren(
        properties.get("vz_nr")
    )

    vz_bez = str(
        properties.get("vz_bez") or ""
    ).lower()


    # --------------------------------------------------------
    # Direkte Geschwindigkeitsbegrenzung
    # --------------------------------------------------------

    if vz_nr == "274":
        return "Geschwindigkeitsbegrenzung"


    # --------------------------------------------------------
    # Tempo-Zonen
    # --------------------------------------------------------

    if vz_nr == "274.1":
        return "Beginn Tempo-Zone"

    if vz_nr == "274.2":
        return "Ende Tempo-Zone"


    # --------------------------------------------------------
    # Ende der Begrenzung
    # --------------------------------------------------------

    if vz_nr == "278":
        return "Ende Geschwindigkeitsbegrenzung"


    # --------------------------------------------------------
    # Fahrradstraße
    # --------------------------------------------------------

    if vz_nr == "244.1":
        return "Beginn Fahrradstraße"

    if vz_nr == "244.2":
        return "Ende Fahrradstraße"


    # --------------------------------------------------------
    # Verkehrsberuhigter Bereich
    # --------------------------------------------------------

    if (
        "verkehrsberuhigter bereich"
        in vz_bez
    ):
        return "Verkehrsberuhigter Bereich"


    # --------------------------------------------------------
    # Fußgängerbereich
    # --------------------------------------------------------

    if (
        "fußgängerbereich"
        in vz_bez
        or "fussgängerbereich"
        in vz_bez
    ):
        return "Fußgängerbereich"


    return "Andere geschwindigkeitsrelevante Regelung"


# ============================================================
# DATEN EINLESEN
# ============================================================

print("Lese Verkehrszeichen-Datei ein...")

with open(
    EINGABEDATEI,
    "r",
    encoding="utf-8"
) as datei:

    daten = json.load(datei)


features = daten.get(
    "features",
    []
)


print(
    f"Anzahl aller Verkehrszeichen: "
    f"{len(features)}"
)


# ============================================================
# RELEVANTE DATEN EXTRAHIEREN
# ============================================================

reduzierte_features = []


for index, feature in enumerate(features):

    properties = feature.get(
        "properties",
        {}
    )

    geometry = feature.get(
        "geometry",
        {}
    )


    # --------------------------------------------------------
    # Nur geschwindigkeitsrelevante Objekte behalten
    # --------------------------------------------------------

    if not ist_geschwindigkeitsrelevant(
        properties
    ):
        continue


    # --------------------------------------------------------
    # Position
    # --------------------------------------------------------

    coordinates = geometry.get(
        "coordinates"
    )


    if not coordinates:
        continue


    # --------------------------------------------------------
    # Geschwindigkeit bestimmen
    # --------------------------------------------------------

    geschwindigkeit = geschwindigkeit_finden(
        properties
    )


    # --------------------------------------------------------
    # Kompakter Datensatz
    # --------------------------------------------------------

    reduzierter_eintrag = {

        # Position des Verkehrszeichens
        "position": coordinates,


        # Straßenzugehörigkeit
        "strasse": properties.get(
            "vz_seg_sn"
        ),


        # Segment-ID
        "segment_id": properties.get(
            "vz_seg_id"
        ),


        # Verkehrszeichen-ID
        "vz_nr": nummer_normalisieren(
            properties.get("vz_nr")
        ),


        # Art der Regelung
        "regelung": regelung_bestimmen(
            properties
        ),


        # Geschwindigkeit in km/h
        "geschwindigkeit_kmh": geschwindigkeit
    }


    reduzierte_features.append(
        reduzierter_eintrag
    )


    # Fortschrittsanzeige
    if index % 100000 == 0:

        print(
            f"Verarbeitet: "
            f"{index} / "
            f"{len(features)}"
        )


# ============================================================
# AUSGABEFORMAT
# ============================================================

ausgabe_daten = {

    "anzahl": len(
        reduzierte_features
    ),

    "daten": reduzierte_features
}


# ============================================================
# DATEI SPEICHERN
# ============================================================

print()
print("Speichere reduzierte Datei...")


with open(
    AUSGABEDATEI,
    "w",
    encoding="utf-8"
) as datei:

    json.dump(
        ausgabe_daten,
        datei,
        ensure_ascii=False,
        separators=(",", ":")
    )


# ============================================================
# ERGEBNIS
# ============================================================

print()
print("=" * 70)
print("DATENREDUKTION ABGESCHLOSSEN")
print("=" * 70)

print(
    f"Verkehrszeichen ursprünglich: "
    f"{len(features)}"
)

print(
    f"Geschwindigkeitsrelevante Objekte: "
    f"{len(reduzierte_features)}"
)

print()
print(
    f"Neue Datei:"
)

print(
    AUSGABEDATEI
)

Lese Verkehrszeichen-Datei ein...
Anzahl aller Verkehrszeichen: 82534

Speichere reduzierte Datei...

DATENREDUKTION ABGESCHLOSSEN
Verkehrszeichen ursprünglich: 82534
Geschwindigkeitsrelevante Objekte: 5539

Neue Datei:
P:\pcloud\Projekte_pcloud\isochrone_map\geschwindigkeitsrelevante_verkehrszeichen.json
